# נתב פניות תמיכה: LLM ראשון בתוך גרף

מודל שפה קורא כל פנייה ומחליט לאיזה צוות היא שייכת: Billing, Tech או General.
התבנית: **node קורא ל־LLM ושומר את ההחלטה ב־state, וקשת מותנית קוראת את ההחלטה ובוחרת את ה־node הבא.**

לפני ההרצה: הוסיפי ב־Colab Secrets (אייקון המפתח בסרגל השמאלי) סוד בשם `OPENAI_API_KEY`, והפעילי את Notebook access.

In [ ]:
!pip -q install langgraph langchain-openai

## מפתח API
המפתח נטען מ־Colab Secrets. הוא לא נשמר במחברת ולא מודפס.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from typing import Literal, TypedDict

import pandas as pd
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

# The model name lives here only. A cheap model is enough to sort tickets into 3 teams.
MODEL = "gpt-6-luna"

## State

In [ ]:
class State(TypedDict):
    ticket: str      # the customer's message (input)
    category: str    # "billing" / "technical" / "general", chosen by the LLM
    team: str        # the team the ticket was forwarded to
    reply: str       # the fixed note sent to the customer

## ה־node שקורא ל־LLM: `classify`
`with_structured_output` מחייב את המודל להחזיר אובייקט `TicketCategory`, ששדה `category` שלו יכול להיות רק אחד משלושת השמות.
`temperature=0` שומר על תשובות עקביות. `gpt-6-luna` הוא מודל reasoning, והוא מקבל `temperature=0` רק כש־`reasoning_effort="none"`.

In [ ]:
class TicketCategory(BaseModel):
    category: Literal["billing", "technical", "general"] = Field(
        description="The team that should handle the ticket")


SYSTEM_PROMPT = """You route customer support tickets for TaskNest, a to-do app.
Read the ticket and choose exactly one category:
- billing: payments, charges, invoices, refunds, prices, subscriptions and plans.
- technical: bugs, crashes, error messages, the app not working as it should, login and account access problems.
- general: everything else, such as feature ideas, how-to questions, feedback and thank-you notes.
Decide by the customer's main problem, not by single words.
If something is broken or shows an error, choose technical, even when it happens while paying."""

# gpt-6-luna is a reasoning model: it accepts temperature=0 only with reasoning turned off.
llm = ChatOpenAI(model=MODEL, temperature=0, reasoning_effort="none").with_structured_output(TicketCategory)


def classify(state: State) -> dict:
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(state["ticket"])]
    # <-- THE LLM CALL
    decision = llm.invoke(messages)
    return {"category": decision.category}

## ה־nodes של הצוותים
Python רגיל, בלי LLM. כל node קובע את הצוות ואת התשובה הקבועה ללקוח.

In [ ]:
def billing_team(state: State) -> dict:
    return {"team": "Billing",
            "reply": "Thanks for reaching out! Your ticket is now with our Billing team. "
                     "They'll get back to you within one business day."}


def tech_team(state: State) -> dict:
    return {"team": "Tech",
            "reply": "Thanks for reaching out! Your ticket is now with our Tech team. "
                     "They'll look into the problem and get back to you soon."}


def general_team(state: State) -> dict:
    return {"team": "General",
            "reply": "Thanks for reaching out! Your ticket is now with our General support team. "
                     "We'll get back to you soon."}

## הקשת המותנית
פונקציית הניתוב לא קוראת ל־LLM. היא רק קוראת את ההחלטה שהמודל כבר כתב ל־state.

In [ ]:
# <-- THE LLM'S ANSWER CHOOSES THE EDGE
def route(state: State) -> Literal["billing", "technical", "general"]:
    return state["category"]

## בניית הגרף

In [ ]:
builder = StateGraph(State)

builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, {
    "billing": "billing_team",
    "technical": "tech_team",
    "general": "general_team",
})
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)

graph = builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## הרצה לדוגמה

In [ ]:
REQUIRED_KEYS = ("ticket", "category", "team", "reply")

result = graph.invoke({"ticket": "I was charged twice this month"})
for key in REQUIRED_KEYS:
    print(f"{key:9}{result[key]}")

## פניות בדיקה
לפחות 2 פניות לכל צוות, 4 פניות עמומות, ו־7 פניות קשות שנכתבו כדי לנסות לבלבל את המודל.
לא בודקים עם `assert` שהמודל בחר את הקטגוריה הצפויה, כי ההחלטה שלו. משווים בטבלה בסוף.

In [ ]:
TEAM_FOR = {"billing": "Billing", "technical": "Tech", "general": "General"}

TESTS = [
    # (kind, ticket, the category I expect)
    ("billing", "I was charged twice this month", "billing"),
    ("billing", "How do I get a refund for my annual subscription?", "billing"),
    ("technical", "The app crashes whenever I open a list with more than 100 tasks", "technical"),
    ("technical", 'I can\'t log in. It says "invalid token" even after I reset my password', "technical"),
    ("general", "It would be great if I could color-code my tasks", "general"),
    ("general", "Just wanted to say thanks, TaskNest keeps my whole week organized!", "general"),
    ("ambiguous", "The app crashes every time I try to pay", "technical"),
    ("ambiguous", "I paid for Premium but the Premium features still don't show up in the app", "technical"),
    ("ambiguous", "Do you have a student discount?", "billing"),
    ("ambiguous", "Thanks for fixing the login bug last week! Any plans for a dark mode?", "general"),
    # Harder tickets, written to try to confuse the LLM
    ("hard", "My invoice shows the wrong company name. Can you fix it?", "billing"),
    ("hard", "I got logged out and now the app asks me to pay again for a plan I already have", "billing"),
    ("hard", "Your new update is terrible. I want my money back.", "billing"),
    ("hard", "How do I export my tasks to a CSV file?", "general"),
    ("hard", "The Upgrade button does nothing when I click it", "technical"),
    ("hard", "האפליקציה לא נפתחת מאז העדכון האחרון", "technical"),
    ("hard", "Is my credit card information safe with you?", "general"),
]

rows = []
for number, (kind, ticket, expected) in enumerate(TESTS, 1):
    result = graph.invoke({"ticket": ticket})

    for key in REQUIRED_KEYS:
        assert key in result, f"#{number}: missing key {key}"
    assert result["category"] in TEAM_FOR, f"#{number}: category {result['category']}"
    assert result["team"] == TEAM_FOR[result["category"]], f"#{number}: team {result['team']}"

    print(f"#{number} ({kind}) {ticket}")
    print(f"   category: {result['category']}")
    print(f"   reply:    {result['reply']}")
    print()

    rows.append({"#": number, "kind": kind, "ticket": ticket, "expected": expected,
                 "llm": result["category"], "team": result["team"],
                 "match": "yes" if result["category"] == expected else "NO"})

print(f"All {len(TESTS)} tickets ran, and every result has the 4 required keys.")

In [ ]:
pd.set_option("display.max_colwidth", None)
summary = pd.DataFrame(rows).set_index("#")
print(f"Matches: {(summary['match'] == 'yes').sum()} / {len(summary)}")
summary